In [1]:
# Код предназначен для классификации текста с использованием метода TF-IDF и модели логистической регрессии. 
# Мы выполняем следующие шаги:
# 
# * Загрузка данных из файлов
# * Предобработка данных
# * Разбиение описаний на предложения
# * Обучение модели TF-IDF и логистической регрессии
# * Классификация тестовых данных
# * Сохранение результатов классификации
# 
# После этого у нас есть два файла с результатами: "clean.xlsx" и "basket.xlsx", в зависимости от значения THRESHOLD.
# 
# **_clean.xlsx_** - сюда попадают примеры после классификации
# **_basket.xlsx_** - сюда попадают примеры которые не были проклассифицированы. Эти примеры нужно изучить и снова проклассифицировать.


import pandas as pd
import numpy as np
from statistics import mean
import re
import nltk
import pymorphy2
from nltk.corpus import stopwords

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)



In [2]:
# Глобальное определение разделителей (Предобработка)
REPLACER = {' г.': ' г', 
            ' р.': ' р',
            '?': '? ',
            '!': '! '}
DELIMIT = ('. ', '? ', '! ')

# Настройка NLTK
nltk.download('stopwords')

# Инициализация морфологического анализатора и стоп-слов
morph = pymorphy2.MorphAnalyzer()
russian_stopwords = stopwords.words("russian")


# Функции предобработки
def remove_stop_words(text):
    return " ".join([word for word in text.split() if word not in russian_stopwords])

def lemmatize_text(text):
    return " ".join([morph.parse(word)[0].normal_form for word in text.split()])

def clean_text(text):
    text = re.sub(r'\d+', '', text)  # Удаляем числа
    text = re.sub(r'[^\w\s]', '', text)  # Удаляем пунктуацию
    return text


def splitter(string):
    string = string.lower().lstrip()
    for key, value in REPLACER.items():
        string = string.replace(key, value)    
    # Компилируем регулярное выражение для повышения производительности
    regexPattern = '|'.join(map(re.escape, DELIMIT))
    compiled_regex = re.compile(regexPattern)
    # Проверка на наличие кириллических символов
    if re.search('[а-яё]', string):
        return list(filter(None, compiled_regex.split(string)))
    else:
        return np.NaN
    
def process_text(text_list):
    # Если результат splitter является списком, объединяем его в строку
    if isinstance(text_list, list):
        return ' '.join(text_list)
    # Если результат splitter является np.NaN (например, пустой список после фильтрации), возвращаем пустую строку
    return ''


def prediction(row, pipeline, label_encoder):
    # Объединение списка предложений в одну строку
    text = ' '.join(row['text']) if isinstance(row['text'], list) else row['text']
    
    # Преобразование текста и получение предсказаний
    response = pipeline['tfidf'].transform([text])
    encoded_result = pipeline['lr'].predict(response)
    
    result = label_encoder.inverse_transform(encoded_result)[0]
    score = np.max(pipeline['lr'].predict_proba(response), axis=1)[0]
    
    # Возвращаем результаты предсказания и вероятности
    return pd.Series({TEST_Y: result, TEST_SCORE: score})

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
INFO:pymorphy2.opencorpora_dict.wrapper:Loading dictionaries from C:\Users\admin\anaconda3\lib\site-packages\pymorphy2_dicts_ru\data
INFO:pymorphy2.opencorpora_dict.wrapper:format: 2.4, revision: 417127, updated: 2020-10-11T15:05:51.070345


In [3]:
# Порог
THRESHOLD = 0.3

# Данные
TRAIN_FILE = 'БК-База-знаний111.xlsx'
TRAIN_X = 'Вопрос'
TRAIN_Y = 'Тип-подкатегория-подподкатегория'

TEST_FILE = 'tickets-August_after_processing.xlsx'
TEST_X = 'opisanie'
TEST_Y = 'Теги'
TEST_SCORE = 'Score'


# Подгрузим наши данные
logger.info('Загрузка данных ...')
train_data = pd.read_excel(TRAIN_FILE, header=0)
test_data = pd.read_excel(TEST_FILE, header=0)
logger.info('Готово')

INFO:__main__:Загрузка данных ...
INFO:__main__:Готово


In [4]:
# Разобьем описание на предложения
logger.info('Разбиваем описание на предложения ...')
#test_data['text'] = test_data[TEST_X].apply(lambda x:splitter(x))
#test_data = test_data.dropna(subset=['text'])
test_data['text'] = test_data[TEST_X].astype(str).apply(lambda x: splitter(x))
logger.info('Готово')

# Проведем предобработку данных
logger.info('Предобработка данных...')
# Применим функцию process_text для обработки результатов splitter, затем проведем остальную предобработку
test_data['text'] = test_data['text'].apply(process_text).apply(clean_text).apply(remove_stop_words).apply(lemmatize_text)
train_data[TRAIN_X] = train_data[TRAIN_X].astype(str).apply(clean_text).apply(remove_stop_words).apply(lemmatize_text)

#train_data[TRAIN_X] = train_data[TRAIN_X].apply(clean_text).apply(remove_stop_words).apply(lemmatize_text)
#test_data[TEST_X] = test_data[TEST_X].astype(str).apply(clean_text).apply(remove_stop_words).apply(lemmatize_text)
#test_data['text'] = test_data['text'].apply(lambda lst: ' '.join(lst)).apply(clean_text).apply(remove_stop_words).apply(lemmatize_text)
logger.info('Предобработка данных завершена')

# Удаление пустых строк после предобработки
train_data = train_data.dropna(subset=[TRAIN_Y])
test_data = test_data.dropna(subset=['text'])

INFO:__main__:Разбиваем описание на предложения ...
INFO:__main__:Готово
INFO:__main__:Предобработка данных...
INFO:__main__:Предобработка данных завершена


In [5]:
# Cоздание экземпляра LabelEncoder
logger.info('Работа LabelEncoder ...')
label_encoder = LabelEncoder()

# Преобразование текстовых меток в строки
train_data[TRAIN_Y] = train_data[TRAIN_Y].apply(str)

# Преобразование текстовых меток в числовые
train_labels_encoded = label_encoder.fit_transform(train_data[TRAIN_Y])

logger.info('Готово')

INFO:__main__:Работа LabelEncoder ...
INFO:__main__:Готово


In [9]:
# Создание pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, max_df=0.75, max_features=20000, 
                              ngram_range=(1, 2), smooth_idf=True, use_idf=True, 
                              sublinear_tf=False, norm='l2')),
    ('lr', LogisticRegression(solver='lbfgs', multi_class='multinomial'))
])

# Обучение модели
logger.info('Обучаем модель ...')
pipeline.fit(train_data[TRAIN_X].values.astype('U'), train_labels_encoded)

logger.info('Модель обучена')

INFO:__main__:Обучаем модель ...
INFO:__main__:Модель обучена


In [7]:
test_data.columns

Index(['ID заявки', 'Теги', 'opisanie', 'text'], dtype='object')

In [8]:
# Произведем классификацию
logger.info('Запуск классификации ...')
# test_data = test_data.apply(lambda row: prediction(row, pipeline), axis=1)
test_data[[TEST_Y, TEST_SCORE]] = test_data.apply(lambda row: prediction(row, pipeline, label_encoder), axis=1)
logger.info('Классификация завершена')

# Удаляем вспомогательную колонку 'text'
test_data = test_data.drop(['text'], axis=1)

# Разделение данных
clean = test_data[test_data[TEST_SCORE]  >= THRESHOLD]
basket = test_data[test_data[TEST_SCORE] < THRESHOLD]

# Сохранение результатов
logger.info('Сохранение результатов ...')
clean.to_excel('clean.xlsx', index=False)
basket.to_excel('basket.xlsx', index=False)
logger.info('Результаты работы модели сохранены')

INFO:__main__:Запуск классификации ...
INFO:__main__:Классификация завершена
INFO:__main__:Сохранение результатов ...
INFO:__main__:Результаты работы модели сохранены
